# VantageGro Retail Stock Insight
## Stage 6: Demand Forecasting & Inventory Optimization Engine

### Objectives
1. **Champion Model Deployment**: Load trained Champion ML model (`HistGradientBoostingRegressor`) with saved feature schema.
2. **Recursive Multi-Step Forecasting**: Project demand 14 days ahead per `(Store ID, Product ID)` series, updating autoregressive lags recursively to avoid future leakage.
3. **What-If Scenario Simulation**: Test impact of planned promotions and price variations on forecasted demand trajectory.
4. **Inventory Optimization Engine**: Formulate and compute Safety Stock ($SS$), Reorder Point ($ROP$), Target Stock Level ($TSL$), and Suggested Reorder Quantity.
5. **Risk Classification**: Categorize each SKU into Critical Stockout Risk, High Stockout Risk, Overstock Risk, or Optimal Stock Level.
6. **Full Fleet Recommendations**: Execute across all 100 store-product series and export prioritized action items to CSV and JSON.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime, timedelta

sns.set_theme(style='whitegrid', palette='mako')
plt.rcParams['figure.figsize'] = (10, 5)
print('Forecasting & Inventory Optimization environment ready!')

### 1. Load Champion ML Model & Historical Feature Matrix

In [ ]:
model_bundle = joblib.load('../models/vantagegro_best_model.joblib')
df_feat = pd.read_csv('../data/processed/vantagegro_features.csv')

model = model_bundle['model']
feature_cols = model_bundle['feature_columns']
print(f"Loaded Champion Model: {model_bundle['model_name']}")
print(f"Number of Features: {len(feature_cols)}")
print(f"Features Matrix Shape: {df_feat.shape}")
print(f"Date Range: {df_feat['Date'].min()} to {df_feat['Date'].max()}")

### 2. Multi-Step Recursive Forecasting Function (Zero Future Leakage)

In [ ]:
def recursive_forecast(
    store_id, product_id, features_df, model_bundle, horizon_days=14, promo_scenario=None
):
    model = model_bundle['model']
    cols = model_bundle['feature_columns']
    series = features_df[(features_df['Store ID'] == store_id) & (features_df['Product ID'] == product_id)].copy()
    series['Date_dt'] = pd.to_datetime(series['Date'])
    series = series.sort_values('Date_dt').reset_index(drop=True)
    
    latest = series.iloc[-1]
    base_date = latest['Date_dt']
    past_demands = series['Demand'].tail(35).tolist()
    
    category_val = latest.get('Category', 'Groceries')
    region_val = latest.get('Region', 'North')
    base_price = float(latest.get('Price', 50.0))
    base_comp = float(latest.get('Competitor Pricing', base_price))
    base_disc = float(latest.get('Discount', 5.0))
    
    steps = []
    for h in range(1, horizon_days + 1):
        step_date = base_date + timedelta(days=h)
        promo_val = promo_scenario if promo_scenario is not None else int(latest.get('Promotion', 0))
        
        feat_dict = {
            'cal_year': step_date.year, 'cal_month': step_date.month, 'cal_day': step_date.day,
            'cal_day_of_week': step_date.dayofweek, 'cal_is_weekend': int(step_date.dayofweek in [5, 6]),
            'cal_week_of_year': int(step_date.isocalendar().week),
            'cal_is_month_start': int(step_date.is_month_start), 'cal_is_month_end': int(step_date.is_month_end),
            'cal_quarter': step_date.quarter,
            'sin_month': round(np.sin(2 * np.pi * step_date.month / 12), 4),
            'cos_month': round(np.cos(2 * np.pi * step_date.month / 12), 4),
            'sin_day_of_week': round(np.sin(2 * np.pi * step_date.dayofweek / 7), 4),
            'cos_day_of_week': round(np.cos(2 * np.pi * step_date.dayofweek / 7), 4),
            'demand_lag_1': past_demands[-1],
            'demand_lag_7': past_demands[-7] if len(past_demands) >= 7 else past_demands[-1],
            'demand_lag_14': past_demands[-14] if len(past_demands) >= 14 else past_demands[-1],
            'demand_lag_28': past_demands[-28] if len(past_demands) >= 28 else past_demands[-1],
            'rolling_mean_7': round(float(np.mean(past_demands[-7:])), 2),
            'rolling_mean_14': round(float(np.mean(past_demands[-14:])), 2),
            'rolling_mean_28': round(float(np.mean(past_demands[-28:])), 2),
            'rolling_std_7': round(float(np.std(past_demands[-7:])), 2),
            'rolling_std_14': round(float(np.std(past_demands[-14:])), 2),
            'recent_demand_trend': round(np.mean(past_demands[-7:]) - np.mean(past_demands[-28:]), 2),
            'Price': base_price, 'Discount': base_disc, 'Promotion': promo_val,
            'Competitor Pricing': base_comp,
            'price_ratio': round(base_price / base_comp, 3) if base_comp > 0 else 1.0,
            'price_diff': round(base_price - base_comp, 2),
            'discounted_price': round(base_price * (1 - base_disc / 100), 2),
            'Inventory Level': float(latest.get('Inventory Level', 100.0)),
            'Units Ordered': float(latest.get('Units Ordered', 0.0)),
            'Epidemic': int(latest.get('Epidemic', 0))
        }
        for c in cols:
            if c not in feat_dict:
                if c.startswith('Category_'):
                    feat_dict[c] = int(c == f'Category_{category_val}')
                elif c.startswith('Region_'):
                    feat_dict[c] = int(c == f'Region_{region_val}')
                elif c.startswith('Weather Condition_'):
                    feat_dict[c] = int(c == 'Weather Condition_Sunny')
                else:
                    feat_dict[c] = 0
        vec = [feat_dict.get(c, 0.0) for c in cols]
        pred = max(0.0, round(float(model.predict([vec])[0]), 2))
        past_demands.append(pred)
        steps.append({'step': h, 'date': step_date.strftime('%Y-%m-%d'), 'predicted_demand': pred, 'promotion': promo_val})
    return steps

# Sample forecast for S001, P0001
s1_fc = recursive_forecast('S001', 'P0001', df_feat, model_bundle, horizon_days=14)
print('Sample 14-Day Forecast (Store S001, SKU P0001):')
print(pd.DataFrame(s1_fc).head(7))

### 3. Scenario Simulation: Promotion Impact Analysis
Compare projected demand trajectory with vs. without promotion.

In [ ]:
fc_baseline = recursive_forecast('S001', 'P0001', df_feat, model_bundle, horizon_days=14, promo_scenario=0)
fc_promo = recursive_forecast('S001', 'P0001', df_feat, model_bundle, horizon_days=14, promo_scenario=1)

df_scenarios = pd.DataFrame({
    'Date': [s['date'] for s in fc_baseline],
    'Baseline Demand (No Promo)': [s['predicted_demand'] for s in fc_baseline],
    'Simulated Promo Demand': [s['predicted_demand'] for s in fc_promo]
})

plt.figure(figsize=(10, 4.5))
plt.plot(df_scenarios['Date'], df_scenarios['Baseline Demand (No Promo)'], marker='o', label='Baseline (No Promo)', color='#3b82f6')
plt.plot(df_scenarios['Date'], df_scenarios['Simulated Promo Demand'], marker='s', label='Scenario: Promotion Active', color='#ef4444', linestyle='--')
plt.title('What-If Scenario Simulation: Promotional Uplift Trajectory (Store S001, P0001)', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Forecasted Daily Demand')
plt.xticks(rotation=30, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

### 4. Inventory Optimization Engine Formulation
- **Lead Time ($L$)**: 3 days
- **Service Level ($z$)**: 1.65 (corresponding to 95.0% non-stockout probability)
- **Review Period ($R$)**: 7 days
- **Demand During Lead Time ($DDLT$)**: $\bar{d} \times L$
- **Safety Stock ($SS$)**: $z \times \sigma_d \times \sqrt{L}$
- **Reorder Point ($ROP$)**: $DDLT + SS$
- **Target Stock Level ($TSL$)**: $\bar{d} \times (L + R) + SS$
- **Suggested Reorder Qty**: $\max(0, TSL - \text{Current Inventory})$
- **Days of Supply (DoS)**: $\frac{\text{Current Inventory}}{\bar{d}}$

In [ ]:
# Load generated full inventory recommendations
df_recs = pd.read_csv('../data/processed/inventory_recommendations.csv')
with open('../data/reports/forecast_and_inventory_report.json') as f:
    report_meta = json.load(f)

print(f"Total Series Evaluated: {len(df_recs)}")
print(f"Risk Breakdown:")
for k, v in report_meta['risk_breakdown'].items():
    print(f"  - {k}: {v} SKUs ({v/len(df_recs)*100:.1f}%)")
print(f"\nTotal Replenishment Units Required: {report_meta['total_suggested_reorder_units']:,} units")
df_recs.head(5)

### 5. High-Risk Replenishment Orders (Action Items)

In [ ]:
high_priority = df_recs[df_recs['Review Priority'] == 'HIGH'].sort_values('Suggested Reorder Qty', ascending=False)
print(f"High Priority Action Items ({len(high_priority)} SKUs):")
print(high_priority[['Store ID', 'Product ID', 'Category', 'Current Inventory', 'Reorder Point', 'Suggested Reorder Qty', 'Days of Supply', 'Risk Category']].head(10))

### 6. Visualizing Inventory Risk Profile & Diagnostic Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Risk Distribution Pie Chart
risk_counts = df_recs['Risk Category'].value_counts()
colors = {'CRITICAL STOCKOUT RISK': '#b91c1c', 'HIGH STOCKOUT RISK': '#ef4444', 'OPTIMAL STOCK LEVEL': '#10b981', 'OVERSTOCK RISK': '#f59e0b'}
axes[0].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', colors=[colors.get(c, '#94a3b8') for c in risk_counts.index], startangle=140)
axes[0].set_title('SKU Risk Distribution Across All Stores', fontweight='bold')

# Top Reorder Quantities
top_orders = high_priority.head(8)
axes[1].bar(top_orders['Product ID'] + ' (' + top_orders['Store ID'] + ')', top_orders['Suggested Reorder Qty'], color='#b91c1c')
axes[1].set_title('Top 8 Urgent Reorder Orders (Units Required)', fontweight='bold')
axes[1].set_ylabel('Order Quantity (Units)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 7. Historical Actual Demand + 14-Day Forecast Trajectory

In [ ]:
s_hist = df_feat[(df_feat['Store ID'] == 'S001') & (df_feat['Product ID'] == 'P0001')].tail(45).copy()
s_hist['Date_dt'] = pd.to_datetime(s_hist['Date'])

df_fc = pd.DataFrame(s1_fc)
df_fc['Date_dt'] = pd.to_datetime(df_fc['date'])

plt.figure(figsize=(11, 4.5))
plt.plot(s_hist['Date_dt'], s_hist['Demand'], label='Historical Actual Demand', color='#3b82f6', linewidth=2)
plt.plot(df_fc['Date_dt'], df_fc['predicted_demand'], label='14-Day Recursive Forecast', color='#10b981', linewidth=2.5, linestyle='--', marker='o')
plt.title('Historical Demand + 14-Day ML Forecast (Store S001, Product P0001)', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Demand Units')
plt.legend()
plt.tight_layout()
plt.show()

### Summary of Stage 6 Artifacts
- Recommendations Table: `data/processed/inventory_recommendations.csv` (100 series)
- Detailed Report: `data/reports/forecast_and_inventory_report.json`
- Visualizations: `data/reports/recommendation_visualizations/` (8 interactive HTML Plotly + PNG + JSON charts)
- Ready for Stage 7: MongoDB Local Database Ingestion & Persistence.